# 第 4 章：線性方程組與高斯消去法

本 Notebook 為精簡對照版，完整教學內容請見 `ch04_linear_systems.md`。

## 學習目標

- 將線性方程組寫成矩陣形式 `Ax = b`，並建立增廣矩陣 `[A|b]`
- 熟悉高斯消去法的三種基本列運算
- 分辨列梯形式 (REF) 與簡化列梯形式 (RREF)
- 手算一個 3 元一次方程組的完整高斯消去過程
- 判斷唯一解、無限多解、無解三種情況
- 用 Python 手刻高斯消去函式，並與 `np.linalg.solve` 交叉驗證

In [1]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)

## 1. 線性方程組的矩陣表示 Ax = b

考慮方程組：

```
2x +  y -  z =   8
-3x -  y + 2z = -11
-2x +  y + 2z =  -3
```

可以寫成 `Ax = b` 的矩陣形式。

In [2]:
A_demo = np.array([
    [2, 1, -1],
    [-3, -1, 2],
    [-2, 1, 2],
], dtype=float)
b_demo = np.array([8, -11, -3], dtype=float)

print("A =")
print(A_demo)
print("b =", b_demo)

A =
[[ 2.  1. -1.]
 [-3. -1.  2.]
 [-2.  1.  2.]]
b = [  8. -11.  -3.]


## 2. 增廣矩陣 [A|b]

把係數矩陣 `A` 與常數向量 `b` 併在一起，就是增廣矩陣。

In [3]:
def augmented_matrix(A, b):
    A = np.array(A, dtype=float)
    b = np.array(b, dtype=float).reshape(-1, 1)
    return np.hstack([A, b])

aug_demo = augmented_matrix(A_demo, b_demo)
print("增廣矩陣 [A|b] =")
print(aug_demo)

增廣矩陣 [A|b] =
[[  2.   1.  -1.   8.]
 [ -3.  -1.   2. -11.]
 [ -2.   1.   2.  -3.]]


## 3. 高斯消去法的三種基本列運算

1. **交換兩列** `R_i <-> R_j`
2. **某列乘上非零常數** `R_i -> k * R_i`（`k ≠ 0`）
3. **某列加上另一列的倍數** `R_i -> R_i + k * R_j`

這三種運算都不會改變方程組的解集合。

## 4. 手算範例：逐步高斯消去

以下逐步示範如何把增廣矩陣化簡為簡化列梯形式 (RREF)。

In [4]:
M = aug_demo.copy()
print("初始增廣矩陣：")
print(M)

print("\n步驟 1: R2 -> R2 + 1.5 R1")
M[1] = M[1] + 1.5 * M[0]
print(M)

print("\n步驟 2: R3 -> R3 + 1.0 R1")
M[2] = M[2] + 1.0 * M[0]
print(M)

print("\n步驟 3: R3 -> R3 - 4 R2 (消去第二欄，達成 REF)")
M[2] = M[2] - (M[2][1] / M[1][1]) * M[1]
print(M)

初始增廣矩陣：
[[  2.   1.  -1.   8.]
 [ -3.  -1.   2. -11.]
 [ -2.   1.   2.  -3.]]

步驟 1: R2 -> R2 + 1.5 R1
[[ 2.   1.  -1.   8. ]
 [ 0.   0.5  0.5  1. ]
 [-2.   1.   2.  -3. ]]

步驟 2: R3 -> R3 + 1.0 R1
[[ 2.   1.  -1.   8. ]
 [ 0.   0.5  0.5  1. ]
 [ 0.   2.   1.   5. ]]

步驟 3: R3 -> R3 - 4 R2 (消去第二欄，達成 REF)
[[ 2.   1.  -1.   8. ]
 [ 0.   0.5  0.5  1. ]
 [ 0.   0.  -1.   1. ]]


In [5]:
print("步驟 4: 主元皆化為 1")
M[0] = M[0] / M[0][0]
M[1] = M[1] / M[1][1]
M[2] = M[2] / M[2][2]
print(M)

print("\n步驟 5: 回代消去，化為 RREF")
M[0] = M[0] - M[0][1] * M[1]
M[1] = M[1] - M[1][2] * M[2]
M[0] = M[0] - M[0][2] * M[2]
print(M)

print("\n解: x =", M[0, -1], ", y =", M[1, -1], ", z =", M[2, -1])
print("(正確解應為 x=2, y=3, z=-1)")

步驟 4: 主元皆化為 1
[[ 1.   0.5 -0.5  4. ]
 [ 0.   1.   1.   2. ]
 [-0.  -0.   1.  -1. ]]

步驟 5: 回代消去，化為 RREF
[[ 1.  0.  0.  2.]
 [ 0.  1.  0.  3.]
 [-0. -0.  1. -1.]]

解: x = 2.0 , y = 3.0 , z = -1.0
(正確解應為 x=2, y=3, z=-1)


## 5. 解的三種情況

- **唯一解**：每個未知數都有對應主元
- **無限多解**：存在自由變數（沒有主元的欄位）
- **無解**：化簡後出現矛盾列 `[0...0 | c]`，`c ≠ 0`

## 6. 手刻高斯消去法函式

以下實作一個通用的 `gaussian_elimination(A, b)` 函式，回傳 RREF、解的類型與解向量。

In [6]:
def gaussian_elimination(A, b, verbose=False):
    """以高斯消去法求解 Ax = b，回傳 RREF、解的類型、與解向量。"""
    A = np.array(A, dtype=float)
    b = np.array(b, dtype=float).reshape(-1, 1)
    M = np.hstack([A, b])
    n_rows, n_cols = M.shape
    n_vars = n_cols - 1

    pivot_row = 0
    pivot_cols = []

    for col in range(n_vars):
        if pivot_row >= n_rows:
            break
        max_row = pivot_row + np.argmax(np.abs(M[pivot_row:, col]))
        if np.isclose(M[max_row, col], 0.0):
            continue
        if max_row != pivot_row:
            M[[pivot_row, max_row]] = M[[max_row, pivot_row]]
        M[pivot_row] = M[pivot_row] / M[pivot_row, col]
        for r in range(n_rows):
            if r != pivot_row and not np.isclose(M[r, col], 0.0):
                M[r] = M[r] - M[r, col] * M[pivot_row]
        if verbose:
            print(f"以 R{pivot_row + 1} 為主元列，消去第 {col + 1} 欄:")
            print(M)
        pivot_cols.append(col)
        pivot_row += 1

    rank_A = len(pivot_cols)
    inconsistent = any(
        np.allclose(M[r, :n_vars], 0.0) and not np.isclose(M[r, -1], 0.0)
        for r in range(n_rows)
    )

    if inconsistent:
        return M, "none", None
    elif rank_A < n_vars:
        return M, "infinite", None
    else:
        x = np.zeros(n_vars)
        for i, col in enumerate(pivot_cols):
            x[col] = M[i, -1]
        return M, "unique", x

In [7]:
rref_result, sol_type, x_result = gaussian_elimination(A_demo, b_demo)
print("RREF:")
print(rref_result)
print("解的類型:", sol_type)
print("解 x =", x_result)

RREF:
[[ 1.  0.  0.  2.]
 [ 0.  1.  0.  3.]
 [ 0.  0.  1. -1.]]
解的類型: unique
解 x = [ 2.  3. -1.]


## 7. 與 `np.linalg.solve` 交叉驗證

In [8]:
x_numpy = np.linalg.solve(A_demo, b_demo)
print("np.linalg.solve 結果 x =", x_numpy)
print("gaussian_elimination 結果 x =", x_result)

is_close = np.allclose(x_numpy, x_result)
print("兩者是否一致 (np.allclose)?", is_close)
assert is_close
print("驗證通過：手刻高斯消去法與 np.linalg.solve 結果一致。")

np.linalg.solve 結果 x = [ 2.  3. -1.]
gaussian_elimination 結果 x = [ 2.  3. -1.]
兩者是否一致 (np.allclose)? True
驗證通過：手刻高斯消去法與 np.linalg.solve 結果一致。


## 8. 三種解的情況範例

### 8.1 唯一解

In [9]:
A_unique = np.array([[1, 1], [2, -1]], dtype=float)
b_unique = np.array([3, 0], dtype=float)
rref_u, type_u, x_u = gaussian_elimination(A_unique, b_unique)
print("RREF [A|b] =")
print(rref_u)
print("解的類型:", type_u)
print("解 x =", x_u, "（意即 x=1, y=2）")
print("np.linalg.solve 驗證:", np.linalg.solve(A_unique, b_unique))

RREF [A|b] =
[[1. 0. 1.]
 [0. 1. 2.]]
解的類型: unique
解 x = [1. 2.] （意即 x=1, y=2）
np.linalg.solve 驗證: [1. 2.]


### 8.2 無限多解（自由變數）

In [10]:
A_inf = np.array([[1, 1, 1], [2, 2, 2]], dtype=float)
b_inf = np.array([6, 12], dtype=float)
rref_inf, type_inf, x_inf = gaussian_elimination(A_inf, b_inf)
print("RREF [A|b] =")
print(rref_inf)
print("解的類型:", type_inf)
print("說明: 只有 1 個主元卻有 3 個未知數，故有 2 個自由變數，解集合為一平面。")

RREF [A|b] =
[[1. 1. 1. 6.]
 [0. 0. 0. 0.]]
解的類型: infinite
說明: 只有 1 個主元卻有 3 個未知數，故有 2 個自由變數，解集合為一平面。


### 8.3 無解（矛盾方程式）

In [11]:
A_none = np.array([[1, 1], [1, 1]], dtype=float)
b_none = np.array([2, 5], dtype=float)
rref_none, type_none, x_none = gaussian_elimination(A_none, b_none)
print("RREF [A|b] =")
print(rref_none)
print("解的類型:", type_none)
print("說明: 化簡後出現 [0, 0 | 3] 矛盾列，無解。")

RREF [A|b] =
[[1. 1. 2.]
 [0. 0. 3.]]
解的類型: none
說明: 化簡後出現 [0, 0 | 3] 矛盾列，無解。


## 重點整理

- 任何線性方程組都可寫成 `Ax = b`，增廣矩陣 `[A|b]` 是高斯消去法的操作對象。
- 高斯消去法只使用三種基本列運算，皆不改變解集合。
- REF 主元階梯排列；RREF 更進一步讓主元為 1 且該欄其餘元素為 0。
- 解只有三種可能：唯一解、無限多解（自由變數）、無解（矛盾列）。
- Python 可用 `np.linalg.solve` 或手刻高斯消去函式求解，兩者結果一致。

更多練習題請見 `ch04_linear_systems.md`。